# Olist 데이터 EDA

적재한 PostgreSQL 데이터를 서비스와 같은 방식(SQL)으로 탐색한다. 목적은 지표 용어집과 골든셋(평가용 질문·정답 SQL)을 만들 때 필요한 사실을 확인하는 것이다.

## 0. 데이터셋 개요
- 출처: Kaggle `olistbr/brazilian-ecommerce` 버전 2. 브라질 마켓플레이스 통합 서비스 Olist의 실제 주문 데이터(2016~2018, 익명화)
- 소규모 판매자의 상품을 대형 마켓플레이스에 올려 주는 구조라, 주문 1건에 여러 판매자의 상품이 섞일 수 있다
- CSV 9개를 테이블 9개로 적재했다

| 테이블 | 1행의 의미 | 키 |
|---|---|---|
| orders | 주문 1건(중심 테이블). 상태와 날짜 5개 | order_id |
| order_items | 주문 안의 상품 1개. 상품가(price)·배송비(freight_value) | order_id + order_item_id |
| order_payments | 결제 1건(카드+바우처처럼 나눠 낼 수 있음) | order_id + payment_sequential |
| order_reviews | 리뷰 1건. 점수 1~5 | review_id + order_id (3절에서 확인) |
| customers | 주문마다 새로 발급되는 고객 ID. 실제 사람은 customer_unique_id | customer_id |
| products | 상품. 카테고리명은 포르투갈어 | product_id |
| product_category_name_translation | 카테고리명 포르투갈어 → 영어 | product_category_name |
| sellers | 판매자와 위치 | seller_id |
| geolocation | 우편번호 앞 5자리별 위경도(같은 값이 여러 행, 키 없음) | - |

분석할 때 주의할 점
- 주문 1건에 상품·결제가 여러 행일 수 있어, order_items와 order_payments를 order_id로 함께 JOIN하면 행이 곱해져 금액이 중복 집계된다
- 재구매처럼 "사람" 기준 분석은 customer_id가 아니라 customer_unique_id로 한다
- 원본 컬럼명 오타(`product_name_lenght`, `product_description_lenght`)를 그대로 유지했다

## 1. 적재 가정 검증
적재 스크립트(`scripts/load_data.py`)는 컬럼 이름 규칙(date·timestamp 포함 또는 `_at`으로 끝남)으로 날짜 컬럼을 골라 변환했다.
데이터셋 문서 기준으로 예상한 날짜 컬럼은 8개다(orders 5, order_items 1, order_reviews 2). 실제 타입으로 확인한다.

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
engine = create_engine(os.environ["DATABASE_URL"])

In [2]:
# 모든 테이블의 컬럼 이름과 타입을 DB 메타데이터에서 조회한다


columns_sql = """
select table_name, column_name, data_type
from information_schema.columns 
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position
"""

columns = pd.read_sql(text(columns_sql), engine)
columns

,table_name,column_name,data_type
0,customers,customer_id,text
1,customers,customer_unique_id,text
2,customers,customer_zip_code_prefix,text
3,customers,customer_city,text
4,customers,customer_state,text
5,geolocation,geolocation_zip_code_prefix,text
6,geolocation,geolocation_lat,double precision
7,geolocation,geolocation_lng,double precision
8,geolocation,geolocation_city,text
9,geolocation,geolocation_state,text


In [3]:
# 날짜 타입으로 저장된 컬럼만 골라 예상한 8개와 비교한다

date_columns_sql = """
select table_name, column_name,data_type
FROM information_schema.columns
where table_schema = 'public' and data_type like 'timestamp%'
order by table_name, ordinal_position
"""

date_columns = pd.read_sql(text(date_columns_sql), engine)
date_columns

,table_name,column_name,data_type
0,order_items,shipping_limit_date,timestamp without time zone
1,order_reviews,review_creation_date,timestamp without time zone
2,order_reviews,review_answer_timestamp,timestamp without time zone
3,orders,order_purchase_timestamp,timestamp without time zone
4,orders,order_approved_at,timestamp without time zone
5,orders,order_delivered_carrier_date,timestamp without time zone
6,orders,order_delivered_customer_date,timestamp without time zone
7,orders,order_estimated_delivery_date,timestamp without time zone


### 발견: 우편번호 앞자리 0 손실 → 수정
첫 적재 때 `*_zip_code_prefix`가 정수(bigint)로 저장돼 `09790`이 `9790`이 됐다(행 수 검증은 통과했지만 값이 바뀜).
`load_data.py`에서 이 컬럼들을 문자열로 읽도록 고친 뒤 재적재했다.
원본 CSV 기준 앞자리가 0인 행 수(customers 23,995 / sellers 1,027 / geolocation 245,733)와 DB 값을 비교해 검증한다.

In [4]:
# 우편번호 앞자리 0이 보존됐는지 테이블별로 센다 (원본 CSV 기준 기대값과 비교)

zip_check_sql = """
select 'customers' as table_name, count(*) as leading_zero_rows 
from customers where customer_zip_code_prefix like '0%'
UNION ALL 
select 'sellers', count(*) from sellers where seller_zip_code_prefix like '0%'
UNION ALL 
select 'geolocation', count(*) from geolocation where geolocation_zip_code_prefix like '0%'
""" 

pd.read_sql(text(zip_check_sql), engine)



,table_name,leading_zero_rows
0,customers,23995
1,sellers,1027
2,geolocation,245733


## 2. 날짜 범위와 결측
골든셋 기간을 정하기 위해 orders의 날짜 5개의 결측·최소·최대와 월별 주문 수를 확인한다.
월별 집계 기준은 구매 시각(`order_purchase_timestamp`)이다.

In [5]:
# orders의 날짜 5개에 대해 결측 수와 가장 이른·늦은 값을 확인한다

date_range_sql = """
select 'order_purchase_timestamp' as column_name,
       count(*) - count(order_purchase_timestamp) as null_rows,
       min(order_purchase_timestamp) as min_value,
       max(order_purchase_timestamp) as max_value
from orders
union all
select 'order_approved_at', count(*) - count(order_approved_at), min(order_approved_at), max(order_approved_at) from orders
union all
select 'order_delivered_carrier_date', count(*) - count(order_delivered_carrier_date), min(order_delivered_carrier_date), max(order_delivered_carrier_date) from orders
union all
select 'order_delivered_customer_date', count(*) - count(order_delivered_customer_date), min(order_delivered_customer_date), max(order_delivered_customer_date) from orders
union all
select 'order_estimated_delivery_date', count(*) - count(order_estimated_delivery_date), min(order_estimated_delivery_date), max(order_estimated_delivery_date) from orders
"""

pd.read_sql(text(date_range_sql), engine)

,column_name,null_rows,min_value,max_value
0,order_purchase_timestamp,0,2016-09-04 21:15:19,2018-10-17 17:30:18
1,order_estimated_delivery_date,0,2016-09-30 00:00:00,2018-11-12 00:00:00
2,order_delivered_carrier_date,1783,2016-10-08 10:34:01,2018-09-11 19:48:28
3,order_approved_at,160,2016-09-15 12:16:38,2018-09-03 17:40:06
4,order_delivered_customer_date,2965,2016-10-11 13:46:32,2018-10-17 13:22:46


In [6]:
# 구매 시각 기준 월별 주문 수로 데이터 시작·끝의 불완전한 달을 찾는다

monthly_orders_sql = """
select date_trunc('month', order_purchase_timestamp) as order_month,
       count(*) as order_count
from orders
group by order_month
order by order_month
"""

monthly_orders = pd.read_sql(text(monthly_orders_sql), engine)
monthly_orders

,order_month,order_count
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


### 결론
- 주문 기간: 구매 시각 기준 2016-09-04 ~ 2018-10-17. 결측 없음
- 2016-09(4건)·2016-10(324건)·2016-12(1건)은 서비스 초기라 적고, 2016-11은 주문이 없어 표에 행이 없다. 2018-09(16건)·2018-10(4건)은 수집이 끝나 가는 구간이다
- → 골든셋과 추이 분석 기간은 **2017-01 ~ 2018-08(20개월)**로 고정한다. 앞뒤 달을 넣으면 실제 감소가 아닌데 급감한 것처럼 보인다
- 결측: 도착일 2,965건, 택배사 인계일 1,783건, 승인일 160건. 배송이 끝나지 않은 주문(취소 등)으로 보인다 → "배송 지연"을 도착일 있는 주문만 대상으로 할지 order_status와 함께 확인해 용어집에서 정한다
- 예상 도착일 최대값(2018-11-12)은 마지막 구매일보다 늦다. 예상일은 미래 날짜일 수 있다